In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# Error analysis: ошибки и неуверенные предсказания

Анализ одной выбранной модели на **val**. Измените EXPERIMENT_ID и запустите сверху вниз. По умолчанию — лучший checkpoint ResNet101.

Полные вероятности читаются из `artifacts/predictions/<EXPERIMENT_ID>/val_probabilities.npz`. Если файла нет, выполняется один val-only inference из `best.pt` без TTA; probabilities сохраняются для следующих запусков. На CPU это может занять время. Ни обучение, ни test не используются; raw и experiments.csv не изменяются.

Для ансамблей, TTA и аудита специалистов используйте `06_tta_ensembles.ipynb`.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

from archive.src.error_analysis import (
    validation_probabilities, probability_cases, plot_probability_cases,
    class_metrics, confusion_pairs, plot_confusion, show_errors, show_train_examples,
)

EXPERIMENT_ID = "resnet101_partial_domain_shift_stronger_001"
RAW_DIR = REPOSITORY_ROOT / "data/raw"
PROBABILITIES_PATH = None  # None: стандартный val_probabilities.npz этого запуска
DEVICE = "auto"  # auto / cpu / cuda
BATCH_SIZE = 16
NUM_WORKERS = 0  # удобно для Windows; на Kaggle можно увеличить
TOP_K = 5  # например, 3 или 5
UNCERTAIN_TOP_N = 30  # можно поставить 50
PAGE_SIZE = 6  # объектов на страницу, у каждого картинка + вероятности


In [ ]:
probabilities, classes = validation_probabilities(
    PROJECT_ROOT, EXPERIMENT_ID, raw_dir=RAW_DIR,
    probabilities_path=PROBABILITIES_PATH, device=DEVICE,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)
scored = probabilities.table(include_scores=True)
PRIMARY = EXPERIMENT_ID
frame = scored.set_index("id").rename(columns={"label": PRIMARY})
print(f"{EXPERIMENT_ID}: {len(scored)} val изображений; "
      f"accuracy={scored.label.eq(scored.true_label).mean():.4%}; "
      f"ошибок={scored.label.ne(scored.true_label).sum()}")


## Ошибки по классам

Recall, precision и пары ошибок показывают, какие классы пропускаем и куда ошибочно относим снимки.


In [ ]:
display(class_metrics(frame, PRIMARY, classes).round(2))
display(confusion_pairs(frame, PRIMARY, classes))
plot_confusion(frame, PRIMARY, classes)
plt.close("all")


## Все ошибочные объекты: картинка и top-k классов

Ошибки отсортированы по убыванию уверенности: сначала уверенные неправильные ответы. Оранжевый — неправильный top-1, зелёный — истинный класс, если он входит в top-k. Вероятность и место истинного класса указаны даже если его нет в top-k. Изображения исходные, без аугментаций.

Softmax — оценка модели, а не доказательство правильности. Для проверки калибровки потребовался бы отдельный анализ.


In [ ]:
errors = probability_cases(probabilities, classes, selection="errors", top_k=TOP_K)
display(errors.drop(columns=["top_classes", "top_probabilities"]))
plot_probability_cases(errors, RAW_DIR / "val", f"Ошибки: {EXPERIMENT_ID}", page_size=PAGE_SIZE)
plt.close("all")


## Самые неоднозначные ответы: минимальный разрыв top-1/top-2

Берём UNCERTAIN_TOP_N объектов **из всей val**, включая правильные и ошибочные. Сортируем по p₁−p₂, а не по разметке или низкой p₁. Чем меньше разрыв, тем ближе два ведущих кандидата. Это относительный top-N: все эти объекты не обязательно имеют почти равные вероятности.


In [ ]:
uncertain = probability_cases(probabilities, classes, selection="uncertain",
                              limit=UNCERTAIN_TOP_N, top_k=TOP_K)
print(f"Показаны {len(uncertain)} объектов; "
      f"среди них ошибок: {(~uncertain.correct).sum()}")
display(uncertain.drop(columns=["top_classes", "top_probabilities"]))
plot_probability_cases(uncertain, RAW_DIR / "val", f"Минимальный top1−top2: {EXPERIMENT_ID}", page_size=PAGE_SIZE)
plt.close("all")


## Необязательные дополнительные просмотры

Фильтр конкретной путаницы и случайные исходные train-примеры. Включайте при необходимости; по умолчанию дополнительные галереи выключены.


In [ ]:
SHOW_EXTRA_GALLERIES = False
if SHOW_EXTRA_GALLERIES:
    show_errors(frame, PRIMARY, classes, RAW_DIR, true_class="church", predicted_class="palace")
    show_train_examples(RAW_DIR, classes, ["church", "palace", "commercial_area"], examples_per_class=3)
    plt.close("all")
